### Step 1: Install Dependencies

**How it works:**  
This block runs a shell command using `!pip` to install essential machine learning, audio, and language libraries. Specifically, it installs `openai-whisper` for Automatic Speech Recognition (ASR), `librosa` and `soundfile` for digital audio processing, `nltk` for natural language and syntax parsing, and `xgboost` for gradient-boosted decision trees.

**Why this was chosen:**  
To perform multimodal feature extraction from the speech audio datasets, we need tools that bridge acoustic analysis (Librosa) with semantic textual transcription (Whisper) and textual structure analysis (NLTK). We install these directly within the environment to establish our baseline working pipeline.

In [2]:
!pip -q install openai-whisper librosa nltk xgboost soundfile

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 17.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


### Step 2: Mount Google Drive

**How it works:**  
Uses the Google Colab API (`google.colab.drive`) to authenticate and mount the user's remote Google Drive filesystem to the local runtime path `/content/drive`.

**Why this was chosen:**  
Since the voice training assets and output directories are stored in the cloud under Google Drive, mounting Drive directly allows fast stream-reading of audio files and persistent caching of the heavy engineered features so we don't have to recalculate them across multiple runtime restarts.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Step 3: Global Imports and Configuration

**How it works:**  
Imports fundamental math libraries (`numpy`, `scipy`), tabular frameworks (`pandas`), OS operations (`pathlib`, `os`), text patterns (`re`), state storage (`pickle`), ASR frameworks (`whisper`), and regression estimators (`RandomForestRegressor`, `ExtraTreesRegressor`). It also checks for CUDA compatibility via PyTorch to accelerate GPU speech processing and sets a global random seed to preserve code reproducibility.

**Why this was chosen:**  
Centralizing library imports at the top of the workflow establishes structured design. Standardizing the environment seed ensures that stochastic modeling steps (such as cross-validation splits and random forest trees) yield identical results when re-run.

In [3]:
import os
import re
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
import nltk
import torch
import whisper

from tqdm.auto import tqdm

from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor
)

from sklearn.model_selection import (
    KFold,
    cross_val_predict
)

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

from scipy.stats import pearsonr, spearmanr

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


### Step 4: Download and Prepare NLTK Tokenizers

**How it works:**  
Downloads linguistic dataset models from the Natural Language Toolkit (NLTK) server (`punkt`, `punkt_tab`, `averaged_perceptron_tagger`, `averaged_perceptron_tagger_eng`). It then imports structural tokenizers and taggers to prepare for syntax extraction.

**Why this was chosen:**  
To compute grammatical features (e.g., word count, noun/verb/adjective densities) from the speech transcripts, NLTK requires localized lookup tables and pre-trained classification models. Downloading these quietly up front prevents thread conflicts or runtime warnings during feature generation loops.

In [4]:
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("averaged_perceptron_tagger", quiet=True)
nltk.download("averaged_perceptron_tagger_eng", quiet=True)

from nltk import word_tokenize
from nltk import sent_tokenize
from nltk import pos_tag

print("NLTK ready")

NLTK ready


### Step 5: Define Dataset Directory Structures

**How it works:**  
Constructs explicit system paths for the training and testing files utilizing Path objects. It also designates a path for a local subdirectory dedicated to caching features (`feature_cache`).

**Why this was chosen:**  
Hardcoding raw paths leads to fragile code. Standardizing paths via `pathlib.Path` ensures platform-agnostic OS compatibility. Creating a dedicated cache subdirectory allows feature dictionaries to be serialized as pickle chunks, protecting progress from accidental disconnection.

In [13]:
from pathlib import Path

DATA_ROOT = Path(
    "/content/drive/MyDrive/data"
)

TRAIN_CSV = DATA_ROOT / "train.csv"
TEST_CSV = DATA_ROOT / "test.csv"
SAMPLE_CSV = DATA_ROOT / "sample_submission.csv"

TRAIN_AUDIO_DIR = DATA_ROOT / "train"
TEST_AUDIO_DIR = DATA_ROOT / "test"

# Cache extracted features in Google Drive
CACHE_DIR = DATA_ROOT / "feature_cache"
CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Train CSV:", TRAIN_CSV)
print("Test CSV:", TEST_CSV)
print("Train audio:", TRAIN_AUDIO_DIR)
print("Test audio:", TEST_AUDIO_DIR)

Train CSV: /content/drive/MyDrive/data/train.csv
Test CSV: /content/drive/MyDrive/data/test.csv
Train audio: /content/drive/MyDrive/data/train
Test audio: /content/drive/MyDrive/data/test


### Step 6: Load Metadata and Inspect Distribution

**How it works:**  
Reads the index CSV datasets containing labels and filenames using Pandas. It prints dataset dimensions (`.shape`), inspects initial records (`.head()`), and prints summary statistics (`.describe()`) for the target label variable.

**Why this was chosen:**  
Exploratory analysis of labels is crucial prior to training. By printing statistical aggregates of the target parameter, we understand the scale, center, and dispersion bounds of our labels, helping us configure model objective metrics.

In [14]:
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

display(train_df.head())

print("\nLabel statistics:")
display(train_df["label"].describe())

Train shape: (769, 2)
Test shape : (216, 2)


,filename,label
0,audio_192.wav,3.0
1,audio_436.wav,3.0
2,audio_447.wav,3.5
3,audio_126.wav,2.0
4,audio_61.wav,2.0



Label statistics:


,label
count,769.000000
mean,3.313394
std,1.239000
min,0.000000
25%,2.500000
50%,3.000000
75%,4.000000
max,5.000000


### Step 7: Integrity Check for Input Files

**How it works:**  
Defines and executes `check_audio_files`, a diagnostic helper function that iterates through index rows to verify the actual existence of referenced files inside our mounted Google Drive.

**Why this was chosen:**  
If any audio assets are missing or corrupted in the source folder, loading them will crash the heavy loop. Performing this preemptive sweep ensures complete alignment between the target table and the storage directory.

In [31]:
def check_audio_files(df, audio_dir, name):

    missing = []
    existing = []

    for filename in df["filename"].astype(str):

        path = Path(audio_dir) / filename

        if path.exists():
            existing.append(filename)
        else:
            missing.append(filename)

    print(
        f"{name}: "
        f"{len(existing)}/{len(df)} files found"
    )

    if missing:
        print("Missing files:")
        print(missing[:20])

    return missing


train_missing = check_audio_files(
    train_df,
    TRAIN_AUDIO_DIR,
    "TRAIN"
)

test_missing = check_audio_files(
    test_df,
    TEST_AUDIO_DIR,
    "TEST"
)

TRAIN: 769/769 files found
TEST: 216/216 files found


### Step 8: Load Whisper Model onto Active Device

**How it works:**  
Loads OpenAI's Whisper model (the 'base' configuration) onto the system GPU (`device=DEVICE`) to enable fast parallel audio transcribing.

**Why this was chosen:**  
Whisper is a state-of-the-art encoder-decoder model optimized for transcribing spoken language. Selecting the `base` variant balances performance and execution speed, allowing quick text extraction without running out of active GPU RAM.

In [32]:
print("Loading Whisper Base...")

whisper_model = whisper.load_model(
    "base",
    device=DEVICE
)

print("Whisper loaded.")

Loading Whisper Base...


100%|████████████████████████████████████████| 139M/139M [00:01<00:00, 133MiB/s]


Whisper loaded.


### Step 9: Transcribe Function Definition

**How it works:**  
Wraps Whisper's model interface into a parameter-controlled `transcribe_audio` utility function. It enforces zero-temperature decoding (greedy selection for optimal accuracy) and disables memory-intensive state-history features.

**Why this was chosen:**  
Abstracting transcription isolates internal parameter tuning. Disabling past contextual memory ensures each clip is interpreted as an independent sample, yielding reliable and reproducible transcriptions.

In [33]:
def transcribe_audio(audio_path):

    result = whisper_model.transcribe(
        str(audio_path),
        language="en",
        fp16=(DEVICE == "cuda"),
        temperature=0,
        condition_on_previous_text=False,
        verbose=False
    )

    text = result.get("text", "").strip()

    segments = result.get("segments", [])

    return text, segments

### Step 10: Extract Semantic NLP Features

**How it works:**  
Splits transcript text into tokens, isolates alphabetic words, runs Parts-Of-Speech (POS) tagging to compute linguistic densities (nouns, verbs, adjectives, prepositions), counts localized filler words, and calculates diversity indices.

**Why this was chosen:**  
Spoken fluency, hesitation metrics, and lexical structure are powerful indicators of grammar and verbal skills. These calculated ratios translate unstructured conversational speech transcripts into structured numerical representations.

In [34]:
FILLER_WORDS = {
    "uh",
    "um",
    "erm",
    "hmm",
    "like",
    "actually",
    "basically",
    "you know",
    "i mean"
}


def safe_div(a, b):

    return float(a) / float(b) if b else 0.0


def extract_nlp_features(text, audio_duration):

    text = text.strip()

    if not text:

        return {
            "total_words": 0,
            "num_nouns": 0,
            "num_verbs": 0,
            "num_adjs": 0,
            "num_advs": 0,
            "num_pronouns": 0,
            "num_prepositions": 0,
            "num_conjunctions": 0,
            "avg_word_len": 0,
            "unique_words": 0,
            "lexical_diversity": 0,
            "num_sentences": 0,
            "avg_words_per_sentence": 0,
            "repeated_word_ratio": 0,
            "filler_count": 0,
            "words_per_second": 0,
            "noun_ratio": 0,
            "verb_ratio": 0,
            "adj_ratio": 0,
            "adv_ratio": 0,
            "pronoun_ratio": 0,
            "preposition_ratio": 0,
            "conjunction_ratio": 0
        }

    tokens = word_tokenize(text.lower())

    words = [
        w for w in tokens
        if re.fullmatch(
            r"[a-zA-Z]+(?:'[a-zA-Z]+)?",
            w
        )
    ]

    if not words:
        words = tokens

    pos_tags = pos_tag(words)

    total_words = len(words)

    num_nouns = sum(
        tag.startswith("NN")
        for _, tag in pos_tags
    )

    num_verbs = sum(
        tag.startswith("VB")
        for _, tag in pos_tags
    )

    num_adjs = sum(
        tag.startswith("JJ")
        for _, tag in pos_tags
    )

    num_advs = sum(
        tag.startswith("RB")
        for _, tag in pos_tags
    )

    num_pronouns = sum(
        tag.startswith("PRP")
        for _, tag in pos_tags
    )

    num_prepositions = sum(
        tag in {"IN", "TO"}
        for _, tag in pos_tags
    )

    num_conjunctions = sum(
        tag == "CC"
        for _, tag in pos_tags
    )

    word_lengths = [
        len(w.replace("'", ""))
        for w in words
    ]

    unique_words = len(set(words))

    lexical_diversity = safe_div(
        unique_words,
        total_words
    )

    sentences = sent_tokenize(text)

    num_sentences = len(sentences)

    avg_words_per_sentence = safe_div(
        total_words,
        num_sentences
    )

    counts = {}

    for word in words:
        counts[word] = counts.get(word, 0) + 1

    repeated_tokens = sum(
        count - 1
        for count in counts.values()
        if count > 1
    )

    repeated_word_ratio = safe_div(
        repeated_tokens,
        total_words
    )

    filler_count = 0

    for filler in FILLER_WORDS:

        filler_count += len(
            re.findall(
                r"\b" + re.escape(filler) + r"\b",
                text
            )
        )

    words_per_second = safe_div(
        total_words,
        audio_duration
    )

    return {

        "total_words": total_words,

        "num_nouns": num_nouns,

        "num_verbs": num_verbs,

        "num_adjs": num_adjs,

        "num_advs": num_advs,

        "num_pronouns": num_pronouns,

        "num_prepositions": num_prepositions,

        "num_conjunctions": num_conjunctions,

        "avg_word_len":
            np.mean(word_lengths)
            if word_lengths else 0,

        "unique_words": unique_words,

        "lexical_diversity":
            lexical_diversity,

        "num_sentences": num_sentences,

        "avg_words_per_sentence":
            avg_words_per_sentence,

        "repeated_word_ratio":
            repeated_word_ratio,

        "filler_count":
            filler_count,

        "words_per_second":
            words_per_second,

        "noun_ratio":
            safe_div(num_nouns, total_words),

        "verb_ratio":
            safe_div(num_verbs, total_words),

        "adj_ratio":
            safe_div(num_adjs, total_words),

        "adv_ratio":
            safe_div(num_advs, total_words),

        "pronoun_ratio":
            safe_div(num_pronouns, total_words),

        "preposition_ratio":
            safe_div(num_prepositions, total_words),

        "conjunction_ratio":
            safe_div(num_conjunctions, total_words)
    }

### Step 11: Extract Speech Timing and Cadence Metrics

**How it works:**  
Analyzes Whisper's token segment timestamps to measure speech duration, pause frequencies, average gap duration (using a threshold of 0.25 seconds), and overall speaking-to-silence ratios.

**Why this was chosen:**  
Pacing, hesitation patterns, and structural pauses provide essential acoustic cues for verbal evaluation. These metrics capture speech cadence independently of the literal words spoken.

In [35]:
def extract_timing_features(
    segments,
    audio_duration
):

    if not segments:

        return {
            "speech_duration": 0,
            "pause_duration": audio_duration,
            "num_pauses": 0,
            "avg_pause": 0,
            "max_pause": 0,
            "speaking_ratio": 0,
            "whisper_segment_count": 0
        }

    segments = sorted(
        [
            s for s in segments
            if s.get("start") is not None
            and s.get("end") is not None
        ],
        key=lambda x: x["start"]
    )

    speech_duration = sum(
        max(
            0,
            float(s["end"]) - float(s["start"])
        )
        for s in segments
    )

    pauses = []

    for previous, current in zip(
        segments[:-1],
        segments[1:]
    ):

        gap = max(
            0,
            float(current["start"])
            - float(previous["end"])
        )

        if gap >= 0.25:
            pauses.append(gap)

    pause_duration = max(
        0,
        audio_duration - speech_duration
    )

    return {

        "speech_duration":
            speech_duration,

        "pause_duration":
            pause_duration,

        "num_pauses":
            len(pauses),

        "avg_pause":
            np.mean(pauses)
            if pauses else 0,

        "max_pause":
            max(pauses)
            if pauses else 0,

        "speaking_ratio":
            safe_div(
                speech_duration,
                audio_duration
            ),

        "whisper_segment_count":
            len(segments)
    }

### Step 12: Extract Audio and Acoustic Features

**How it works:**  
Loads raw audio files via Librosa to compute root-mean-square energy (RMS), zero-crossing rates, spectral centroids, spectral rolloff, MFCC banks, chromagram structures, tempo (BPM), and fundamental pitch averages (YIN algorithm).

**Why this was chosen:**  
Linguistic indicators are only half of the story. Physical sound attributes (such as pitch stability, timber, volume variations, and harmonic distribution) represent important vocal markers needed for robust modeling.

In [36]:
def extract_librosa_features(
    audio_path,
    sr=16000
):

    y, sr = librosa.load(
        audio_path,
        sr=sr,
        mono=True
    )

    if len(y) == 0:
        raise ValueError("Empty audio")

    duration = len(y) / sr

    rms = librosa.feature.rms(y=y)[0]

    zcr = librosa.feature.zero_crossing_rate(y)[0]

    centroid = librosa.feature.spectral_centroid(
        y=y,
        sr=sr
    )[0]

    bandwidth = librosa.feature.spectral_bandwidth(
        y=y,
        sr=sr
    )[0]

    rolloff = librosa.feature.spectral_rolloff(
        y=y,
        sr=sr
    )[0]

    mfcc = librosa.feature.mfcc(
        y=y,
        sr=sr,
        n_mfcc=13
    )

    chroma = librosa.feature.chroma_stft(
        y=y,
        sr=sr
    )

    contrast = librosa.feature.spectral_contrast(
        y=y,
        sr=sr
    )

    tempo = librosa.beat.beat_track(
        y=y,
        sr=sr
    )[0]

    tempo = float(
        np.asarray(tempo).reshape(-1)[0]
    )

    # Approximate speech pitch
    try:

        f0 = librosa.yin(
            y,
            fmin=70,
            fmax=400,
            sr=sr
        )

        f0 = f0[np.isfinite(f0)]

        f0 = f0[
            (f0 >= 70) &
            (f0 <= 400)
        ]

    except Exception:

        f0 = np.array([])

    features = {

        "audio_duration":
            duration,

        "rms_mean":
            rms.mean(),

        "rms_std":
            rms.std(),

        "zcr_mean":
            zcr.mean(),

        "zcr_std":
            zcr.std(),

        "spectral_centroid_mean":
            centroid.mean(),

        "spectral_centroid_std":
            centroid.std(),

        "spectral_bandwidth_mean":
            bandwidth.mean(),

        "spectral_bandwidth_std":
            bandwidth.std(),

        "spectral_rolloff_mean":
            rolloff.mean(),

        "spectral_rolloff_std":
            rolloff.std(),

        "tempo":
            tempo,

        "pitch_mean":
            f0.mean()
            if len(f0) else 0,

        "pitch_std":
            f0.std()
            if len(f0) else 0
    }

    # MFCC
    for i in range(13):

        features[
            f"mfcc_{i+1}_mean"
        ] = mfcc[i].mean()

        features[
            f"mfcc_{i+1}_std"
        ] = mfcc[i].std()

    # Chroma
    for i in range(12):

        features[
            f"chroma_{i+1}_mean"
        ] = chroma[i].mean()

        features[
            f"chroma_{i+1}_std"
        ] = chroma[i].std()

    # Spectral contrast
    for i in range(
        contrast.shape[0]
    ):

        features[
            f"contrast_{i+1}_mean"
        ] = contrast[i].mean()

        features[
            f"contrast_{i+1}_std"
        ] = contrast[i].std()

    return features

### Step 13: Feature Fusion Orchestrator

**How it works:**  
Combines NLP text features, segment timing structures, and Librosa acoustic feature outputs into a unified dictionary representing a single audio sample.

**Why this was chosen:**  
This wrapper creates a single, clean interface for complete feature extraction, making the main dataset processing loops much simpler and easier to maintain.

In [37]:
def extract_all_features(audio_path):

    audio_path = Path(audio_path)

    y, sr = librosa.load(
        audio_path,
        sr=16000,
        mono=True
    )

    duration = (
        len(y) / sr
        if len(y)
        else 0
    )

    # Whisper
    text, segments = transcribe_audio(
        audio_path
    )

    # NLP
    nlp = extract_nlp_features(
        text,
        duration
    )

    # Timing
    timing = extract_timing_features(
        segments,
        duration
    )

    # Librosa
    audio = extract_librosa_features(
        audio_path
    )

    features = {}

    features.update(nlp)
    features.update(timing)
    features.update(audio)

    # Keep transcript for inspection
    features["text"] = text

    return features

### Step 14: Local Test on Sample File

**How it works:**  
Runs the entire multimodal extraction pipeline on a single validation file, printing the output text and total feature counts for manual review.

**Why this was chosen:**  
Testing on a single sample lets us verify the entire pipeline—from audio decoding and transcription to acoustic feature extraction—before launching long, multi-hour loops across the entire dataset.

In [38]:
sample_filename = train_df.iloc[0]["filename"]

sample_path = (
    TRAIN_AUDIO_DIR /
    sample_filename
)

sample_features = extract_all_features(
    sample_path
)

print(
    "Transcript:\n",
    sample_features["text"][:500]
)

print(
    "\nNumber of features:",
    len(sample_features)
)

100%|██████████| 6050/6050 [00:04<00:00, 1440.19frames/s]


Transcript:
 Well, this petite land is very, very beautiful and fun. There are fleets like the navy around the swing. The swing usually enjoys this swing. it has to be like it's more like a game of pay, because once it's in it and the other thing, we'll have to finish and the miracle route which is fine for me and most of them play for on some parts why others use these things, some of the people like to do some classes and most times, he had the sound of What do you mean children, last in kid?

Number of features: 109


### Step 15: Parallel Dataset Processor Loop

**How it works:**  
Defines a helper function `process_dataset` that loops through all rows, searches for cached `.pkl` results to avoid redundant calculations, and generates new features only when no cache exists.

**Why this was chosen:**  
Caching is essential when running expensive cloud operations. Saving feature dictionaries individually ensures we don't lose progress if the runtime disconnects, while safely capturing any failed files in a separate list.

In [39]:
def process_dataset(
    df,
    audio_dir,
    split_name
):

    rows = []
    failed = []

    for _, row in tqdm(
        df.iterrows(),
        total=len(df),
        desc=split_name
    ):

        filename = str(
            row["filename"]
        )

        audio_path = (
            Path(audio_dir) /
            filename
        )

        cache_path = (
            CACHE_DIR /
            f"{split_name}_{Path(filename).stem}.pkl"
        )

        try:

            # Load cached features
            if cache_path.exists():

                with open(
                    cache_path,
                    "rb"
                ) as f:

                    features = pickle.load(f)

            else:

                features = extract_all_features(
                    audio_path
                )

                with open(
                    cache_path,
                    "wb"
                ) as f:

                    pickle.dump(
                        features,
                        f
                    )

            features = dict(features)

            features["filename"] = filename

            if "label" in row:

                features["label"] = float(
                    row["label"]
                )

            rows.append(features)

        except Exception as e:

            print(
                f"FAILED: {filename} -> {e}"
            )

            failed.append(
                (filename, str(e))
            )

    return (
        pd.DataFrame(rows),
        failed
    )

### Step 16: Extract and Cache Training Features

**How it works:**  
Runs `process_dataset` across the training split, saving the extracted feature sets as cached files on Google Drive.

**Why this was chosen:**  
This executes our pipeline to compile the full training dataset. Running feature extraction here prepares the structured tabular data needed for downstream model training.

In [40]:
train_feat_df, train_failed = process_dataset(
    train_df,
    TRAIN_AUDIO_DIR,
    "train"
)

print(
    "Training feature shape:",
    train_feat_df.shape
)

print(
    "Failed:",
    len(train_failed)
)

display(
    train_feat_df.head()
)

train:   0%|          | 0/769 [00:00<?, ?it/s]


100%|██████████| 6050/6050 [00:01<00:00, 3534.90frames/s]

100%|██████████| 6007/6007 [00:01<00:00, 4370.87frames/s]

100%|██████████| 6007/6007 [00:01<00:00, 3455.67frames/s]

100%|██████████| 4506/4506 [00:01<00:00, 3612.01frames/s]

100%|██████████| 4530/4530 [00:01<00:00, 3483.16frames/s]

100%|██████████| 6007/6007 [00:01<00:00, 3805.09frames/s]

100%|██████████| 5216/5216 [00:01<00:00, 4043.70frames/s]

100%|██████████| 6007/6007 [00:01<00:00, 3759.02frames/s]

100%|██████████| 6007/6007 [00:01<00:00, 4590.73frames/s]

100%|██████████| 6007/6007 [00:01<00:00, 3613.83frames/s]

100%|██████████| 6007/6007 [00:01<00:00, 3531.74frames/s]

100%|██████████| 6007/6007 [00:01<00:00, 4313.77frames/s]

100%|██████████| 6007/6007 [00:00<00:00, 6582.42frames/s]

100%|██████████| 6007/6007 [00:01<00:00, 4911.21frames/s]

100%|██████████| 6007/6007 [00:01<00:00, 3518.24frames/s]

100%|██████████| 4506/4506 [00:01<00:00, 4092.36frames/s]

100%|██████████| 2653/2653 [00:00<00:00, 4532.84frames/

Training feature shape: (769, 111)
Failed: 0


,total_words,num_nouns,num_verbs,num_adjs,num_advs,num_pronouns,num_prepositions,num_conjunctions,avg_word_len,unique_words,...,contrast_4_std,contrast_5_mean,contrast_5_std,contrast_6_mean,contrast_6_std,contrast_7_mean,contrast_7_std,text,filename,label
0,98,18,17,6,6,9,19,5,3.785714,66,...,3.210733,17.639578,2.802836,29.395510,8.226018,14.700777,2.247479,"Well, this petite land is very, very beautiful...",audio_192.wav,3.0
1,114,29,13,9,9,7,18,7,4.271930,56,...,5.091505,22.759008,5.267385,25.445152,6.730892,17.154606,3.399397,"To see no of a school playground, our playgrou...",audio_436.wav,3.0
2,146,39,30,12,5,4,24,5,3.972603,74,...,4.271398,20.881220,5.042710,21.672067,4.238821,18.943922,4.355758,I favorite place to visit is the beach. I love...,audio_447.wav,3.5
3,99,21,13,2,2,13,24,4,3.636364,45,...,4.378150,19.122998,4.271698,20.789503,5.716175,17.140173,2.848577,"Once upon a time, we, with five friends, went ...",audio_126.wav,2.0
4,76,19,17,5,5,4,11,2,3.750000,47,...,4.663063,19.890432,4.597918,22.964837,6.782134,21.475171,5.674760,"Well, when I was a child, I remember I went to...",audio_61.wav,2.0


### Step 17: Extract and Cache Test Features

**How it works:**  
Runs the feature extraction pipeline on the unseen test split, applying the same caching strategy used for the training data.

**Why this was chosen:**  
Extracting test features with the exact same functions prevents train-test skew, ensuring our test set features are computed identically to our training data.

In [41]:
test_feat_df, test_failed = process_dataset(
    test_df,
    TEST_AUDIO_DIR,
    "test"
)

print(
    "Test feature shape:",
    test_feat_df.shape
)

print(
    "Failed:",
    len(test_failed)
)

display(
    test_feat_df.head()
)

test:   0%|          | 0/216 [00:00<?, ?it/s]


100%|██████████| 4506/4506 [00:01<00:00, 4160.11frames/s]

100%|██████████| 4496/4496 [00:01<00:00, 4438.33frames/s]

100%|██████████| 4501/4501 [00:01<00:00, 3351.57frames/s]

100%|██████████| 5898/5898 [00:00<00:00, 6333.66frames/s]

100%|██████████| 4504/4504 [00:01<00:00, 3204.38frames/s]

100%|██████████| 4506/4506 [00:01<00:00, 3121.58frames/s]

100%|██████████| 4484/4484 [00:01<00:00, 3309.84frames/s]

100%|██████████| 4506/4506 [00:00<00:00, 5379.12frames/s]

100%|██████████| 4506/4506 [00:00<00:00, 4737.69frames/s]

100%|██████████| 4486/4486 [00:01<00:00, 2453.42frames/s]

100%|██████████| 6008/6008 [00:01<00:00, 3633.32frames/s]

100%|██████████| 4506/4506 [00:00<00:00, 4928.20frames/s]

100%|██████████| 4506/4506 [00:01<00:00, 3763.12frames/s]

100%|██████████| 1856/1856 [00:00<00:00, 4167.69frames/s]

100%|██████████| 6008/6008 [00:01<00:00, 4176.68frames/s]

100%|██████████| 4470/4470 [00:01<00:00, 2373.01frames/s]

100%|██████████| 4599/4599 [00:01<00:00, 3534.91frames/

Test feature shape: (216, 111)
Failed: 0


,total_words,num_nouns,num_verbs,num_adjs,num_advs,num_pronouns,num_prepositions,num_conjunctions,avg_word_len,unique_words,...,contrast_4_std,contrast_5_mean,contrast_5_std,contrast_6_mean,contrast_6_std,contrast_7_mean,contrast_7_std,text,filename,label
0,95,16,15,8,5,13,19,5,3.736842,49,...,4.688382,17.819392,3.774545,21.071540,5.607544,15.303754,1.718090,The moment in my life that I cherish the most ...,audio_128.wav,-1.0
1,69,23,9,5,2,5,11,6,5.724638,57,...,5.218183,21.848937,3.792790,21.395203,5.911781,18.657817,2.310826,One of the role models is someone who embodies...,audio_156.wav,-1.0
2,102,20,16,8,11,15,17,4,4.411765,66,...,4.189961,18.901296,4.828663,22.679621,5.279370,19.070777,4.333795,"I guess my favorite sport would be Taekwondo, ...",audio_19.wav,-1.0
3,58,15,13,3,2,9,5,7,4.517241,45,...,3.646279,16.783989,2.801471,16.431319,1.974656,15.797747,1.986251,Hospital having more of doctors and there are ...,audio_108.wav,-1.0
4,87,20,18,6,5,2,8,9,3.770115,52,...,4.410543,19.934661,3.900578,18.372932,3.117998,17.248887,3.087268,"I remember when I went to Goa for a trip, then...",audio_206.wav,-1.0


### Step 18: Save Consolidated Features to Disk

**How it works:**  
Converts the collected feature dictionaries into CSV tables and writes them directly to Google Drive.

**Why this was chosen:**  
Saving features to permanent CSV tables means we can instantly load our tabular datasets in future sessions without needing to mount the audio files or run any deep learning pipelines.

In [42]:
train_feat_df.to_csv(
    CACHE_DIR / "train_features.csv",
    index=False
)

test_feat_df.to_csv(
    CACHE_DIR / "test_features.csv",
    index=False
)

print("Feature tables saved.")

Feature tables saved.


### Step 19: Data Preprocessing and Imputation

**How it works:**  
Filters out non-numeric fields, converts remaining features to numerical types, replaces infinity values with NaNs, and fills missing values using training set medians.

**Why this was chosen:**  
Machine learning models require clean, complete numerical matrices. Imputing missing values with the training set median prevents data leakage from the test set, ensuring our model trains on valid, realistic distributions.

In [43]:
ORIGINAL_FEATURES = [
    "total_words",
    "num_nouns",
    "num_verbs",
    "num_adjs",
    "avg_word_len"
]

FEATURE_COLUMNS = [
    col
    for col in train_feat_df.columns
    if col not in {
        "filename",
        "label",
        "text"
    }
]

X = train_feat_df[
    FEATURE_COLUMNS
].copy()

X_test = test_feat_df[
    FEATURE_COLUMNS
].copy()

y = train_feat_df[
    "label"
].astype(float)

# Numeric conversion
X = X.apply(
    pd.to_numeric,
    errors="coerce"
)

X_test = X_test.apply(
    pd.to_numeric,
    errors="coerce"
)

# Replace infinity
X = X.replace(
    [np.inf, -np.inf],
    np.nan
)

X_test = X_test.replace(
    [np.inf, -np.inf],
    np.nan
)

# Median imputation
train_medians = X.median()

X = X.fillna(
    train_medians
)

X_test = X_test.fillna(
    train_medians
)

# Original notebook features
X_original = train_feat_df[
    ORIGINAL_FEATURES
].copy()

X_original = X_original.apply(
    pd.to_numeric,
    errors="coerce"
)

X_original = X_original.fillna(
    X_original.median()
)

print(
    "Original feature count:",
    X_original.shape[1]
)

print(
    "Fused feature count:",
    X.shape[1]
)

Original feature count: 5
Fused feature count: 108


### Step 20: Performance Evaluation Utility

**How it works:**  
Defines a helper function `evaluate_predictions` to calculate and print key regression metrics, including Root Mean Squared Error (RMSE), Mean Absolute Error (MAE), R-squared ($R^2$), Pearson correlation, and Spearman correlation.

**Why this was chosen:**  
Standardizing our evaluation metrics in a single function ensures we compare all models fairly and consistently using the exact same criteria.

In [44]:
def evaluate_predictions(
    y_true,
    predictions,
    model_name
):

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            predictions
        )
    )

    mae = mean_absolute_error(
        y_true,
        predictions
    )

    r2 = r2_score(
        y_true,
        predictions
    )

    pearson = pearsonr(
        y_true,
        predictions
    )[0]

    spearman = spearmanr(
        y_true,
        predictions
    ).statistic

    result = {

        "model": model_name,

        "RMSE": rmse,

        "MAE": mae,

        "R2": r2,

        "Pearson": pearson,

        "Spearman": spearman
    }

    print(
        f"\n{model_name}"
    )

    print(
        "RMSE:",
        round(rmse, 4)
    )

    print(
        "MAE:",
        round(mae, 4)
    )

    print(
        "R2:",
        round(r2, 4)
    )

    print(
        "Pearson:",
        round(pearson, 4)
    )

    print(
        "Spearman:",
        round(spearman, 4)
    )

    return result

### Step 21: Setup Cross-Validation Strategy

**How it works:**  
Initializes a 5-fold cross-validation scheme using `KFold` with data shuffling enabled and a fixed random state.

**Why this was chosen:**  
Shuffled 5-fold cross-validation provides a robust, unbiased estimate of model performance across the entire dataset, helping to prevent overfitting and guide model selection.

In [45]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("5-fold CV ready.")

5-fold CV ready.


### Step 22: Baseline Model Training (Original Features)

**How it works:**  
Fits a Random Forest Regressor using only the 5 original features and evaluates its performance via out-of-fold (OOF) cross-validation.

**Why this was chosen:**  
Establishing a baseline on the original feature set gives us a clear reference point, allowing us to measure exactly how much performance improves when adding our new multimodal features.

In [46]:
rf_original = RandomForestRegressor(
    n_estimators=500,
    random_state=42,
    n_jobs=-1,
    max_features=1.0,
    min_samples_leaf=2
)

oof_original = cross_val_predict(
    rf_original,
    X_original,
    y,
    cv=cv,
    n_jobs=1
)

result_original = evaluate_predictions(
    y,
    oof_original,
    "Original 5-feature Random Forest"
)


Original 5-feature Random Forest
RMSE: 1.0774
MAE: 0.8572
R2: 0.2429
Pearson: 0.5
Spearman: 0.4313


### Step 23: Fused Features Random Forest Model

**How it works:**  
Trains a Random Forest Regressor on the complete set of 108 fused multimodal features and records cross-validation scores.

**Why this was chosen:**  
This step tests how well a standard Random Forest model handles our expanded feature space, letting us directly compare it against our baseline model.

In [47]:
rf_fused = RandomForestRegressor(
    n_estimators=700,
    random_state=42,
    n_jobs=-1,
    max_features=0.7,
    min_samples_leaf=2
)

oof_rf = cross_val_predict(
    rf_fused,
    X,
    y,
    cv=cv,
    n_jobs=1
)

result_rf = evaluate_predictions(
    y,
    oof_rf,
    "Fused Random Forest"
)


Fused Random Forest
RMSE: 0.7425
MAE: 0.5868
R2: 0.6404
Pearson: 0.8022
Spearman: 0.7005


### Step 24: Fused Features Extra Trees Model

**How it works:**  
Trains and evaluates an Extra Trees Regressor using out-of-fold cross-validation on our full multimodal feature set.

**Why this was chosen:**  
Extra Trees adds extra randomness when choosing decision splits, which can improve generalization and reduce variance when working with large, complex feature sets.

In [48]:
extra_trees = ExtraTreesRegressor(
    n_estimators=700,
    random_state=42,
    n_jobs=-1,
    max_features=0.8,
    min_samples_leaf=2
)

oof_extra = cross_val_predict(
    extra_trees,
    X,
    y,
    cv=cv,
    n_jobs=1
)

result_extra = evaluate_predictions(
    y,
    oof_extra,
    "Fused Extra Trees"
)


Fused Extra Trees
RMSE: 0.7269
MAE: 0.5759
R2: 0.6554
Pearson: 0.8128
Spearman: 0.7227


### Step 25: Fused Features XGBoost Model

**How it works:**  
Trains and evaluates an XGBoost Regressor using our full multimodal feature set and 5-fold cross-validation.

**Why this was chosen:**  
Gradient boosting models like XGBoost are highly effective at finding non-linear relationships in tabular data, often delivering top-tier performance on complex features.

In [49]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    n_estimators=600,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

oof_xgb = cross_val_predict(
    xgb_model,
    X,
    y,
    cv=cv,
    n_jobs=1
)

result_xgb = evaluate_predictions(
    y,
    oof_xgb,
    "Fused XGBoost"
)


Fused XGBoost
RMSE: 0.7074
MAE: 0.5495
R2: 0.6736
Pearson: 0.8211
Spearman: 0.7309


### Step 26: Model Comparison

**How it works:**  
Combines the cross-validation metrics from all models into a single DataFrame and sorts them by RMSE.

**Why this was chosen:**  
Consolidating and sorting our results makes it easy to spot our top-performing model and compare performance gains across different approaches.

In [50]:
results = pd.DataFrame([
    result_original,
    result_rf,
    result_extra,
    result_xgb
])

results = results.sort_values(
    "RMSE"
).reset_index(drop=True)

display(results)

,model,RMSE,MAE,R2,Pearson,Spearman
0,Fused XGBoost,0.707374,0.549520,0.673622,0.821136,0.730908
1,Fused Extra Trees,0.726854,0.575930,0.655399,0.812832,0.722659
2,Fused Random Forest,0.742531,0.586848,0.640373,0.802168,0.700454
3,Original 5-feature Random Forest,1.077385,0.857182,0.242881,0.499968,0.431255


### Step 27: Select the Best Model

**How it works:**  
Identifies and extracts the name of our top-performing model from our sorted results table.

**Why this was chosen:**  
Automating model selection ensures we always choose the mathematically optimal model to generate our final test predictions.

In [51]:
best_model_name = results.iloc[0]["model"]

print(
    "Best model:",
    best_model_name
)

Best model: Fused XGBoost


### Step 28: Train Final Model on Full Dataset

**How it works:**  
Initializes the selected best-performing model configuration and trains it on the entire training dataset.

**Why this was chosen:**  
Training on the full dataset maximizes our data utilization, helping our final model generate the most accurate predictions possible for the test set.

In [53]:
if best_model_name == "Original 5-feature Random Forest":

    final_model = RandomForestRegressor(
        n_estimators=700,
        random_state=42,
        n_jobs=-1,
        max_features=1.0,
        min_samples_leaf=2
    )

    final_X = X_original

    final_X_test = test_feat_df[
        ORIGINAL_FEATURES
    ].copy()

    final_X_test = final_X_test.apply(
        pd.to_numeric,
        errors="coerce"
    )

    final_X_test = final_X_test.fillna(
        X_original.median()
    )


elif best_model_name == "Fused Extra Trees":

    final_model = ExtraTreesRegressor(
        n_estimators=900,
        random_state=42,
        n_jobs=-1,
        max_features=0.8,
        min_samples_leaf=2
    )

    final_X = X
    final_X_test = X_test


elif best_model_name == "Fused XGBoost":

    final_model = XGBRegressor(
        n_estimators=600,
        max_depth=4,
        learning_rate=0.03,
        subsample=0.85,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    final_X = X
    final_X_test = X_test


else:

    final_model = RandomForestRegressor(
        n_estimators=900,
        random_state=42,
        n_jobs=-1,
        max_features=0.7,
        min_samples_leaf=2
    )

    final_X = X
    final_X_test = X_test


final_model.fit(
    final_X,
    y
)

print(
    "Final model trained."
)

Final model trained.


### Step 29: Generate and Clip Test Predictions

**How it works:**  
Generates predictions on our test set and clips the outputs to our valid target range of [0, 5].

**Why this was chosen:**  
Clipping outputs prevents our regression model from predicting values outside of our target scale, ensuring all final scores are realistic and valid.

In [54]:
test_predictions = final_model.predict(
    final_X_test
)

# Valid grammar score range
test_predictions = np.clip(
    test_predictions,
    0,
    5
)

print(
    "Predictions:",
    len(test_predictions)
)

print(
    "Min:",
    test_predictions.min()
)

print(
    "Max:",
    test_predictions.max()
)

print(
    "Mean:",
    test_predictions.mean()
)

Predictions: 216
Min: 2.0925145
Max: 5.0
Mean: 3.2622364


### Step 30: Format and Save Submission File

**How it works:**  
Builds a structured submission DataFrame, runs integrity checks to confirm file names match, and exports the final results as a CSV file.

**Why this was chosen:**  
Adding validation asserts ensures our submission matches all requirements and formatting rules before we finalize and download our predictions.

In [55]:
submission = pd.DataFrame({

    "filename":
        test_df["filename"].astype(str).values,

    "label":
        test_predictions

})

assert len(submission) == len(test_df)

assert (
    submission["filename"].tolist()
    ==
    test_df["filename"].astype(str).tolist()
)

assert submission["label"].between(
    0,
    5
).all()

SUBMISSION_PATH = "/content/submission.csv"

submission.to_csv(
    SUBMISSION_PATH,
    index=False
)

display(
    submission.head(20)
)

print(
    "Submission saved:",
    SUBMISSION_PATH
)

print(
    "Rows:",
    len(submission)
)

,filename,label
0,audio_128.wav,2.778079
1,audio_156.wav,3.926936
2,audio_19.wav,3.517463
3,audio_108.wav,2.398384
4,audio_206.wav,2.965653
5,audio_161.wav,2.745727
6,audio_215.wav,3.104578
7,audio_213.wav,2.599856
8,audio_11.wav,3.510303
9,audio_155.wav,2.893670


Submission saved: /content/submission.csv
Rows: 216


### Step 31: Download Submission File

**How it works:**  
Uses Google Colab's file helper module to prompt a direct browser download of the final `submission.csv` file.

**Why this was chosen:**  
This step triggers a standard download prompt, making it easy to save your final submission file locally with a single click.

In [61]:
from google.colab import files

files.download(
    "/content/submission.csv"
)

print("file downloaded")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

file downloaded


### Step 32: Save Final Model Pipeline

**How it works:**  
Serializes the trained final model, feature definitions, and configuration metadata into a single `.pkl` file on Google Drive.

**Why this was chosen:**  
Saving the entire pipeline as a pickle file allows you to instantly reload, share, or deploy your fully-trained model in future projects without repeating any training steps.

In [60]:
MODEL_PATH = "/content/drive/MyDrive/data/best_model.pkl"

with open(
    MODEL_PATH,
    "wb"
) as f:

    pickle.dump(
        {
            "model": final_model,
            "feature_columns": FEATURE_COLUMNS,
            "original_features": ORIGINAL_FEATURES,
            "selected_model": best_model_name
        },
        f
    )

print(
    "Model saved:",
    MODEL_PATH
)

Model saved: /content/drive/MyDrive/data/best_model.pkl
